In [88]:
using LowLevelFEM
using SparseArrays
using LinearAlgebra

In [89]:
struct Problem
    name::String
    type::Symbol
    dim::Int64
    pdim::Int64
    material::Vector{Material}
    thickness::Float64
    non::Int64
    geometry::LowLevelFEM.Geometry
    field::Symbol
    rhs_field::Symbol
    Problem() = new()
    Problem(name, type, dim, pdim, material, thickness, non, geometry, field, rhs_field) =
        new(name, type, dim, pdim, material, thickness, non, geometry, field, rhs_field)
    Problem(name, type, dim, pdim, material, thickness, non, geometry) =
        new(name, type, dim, pdim, material, thickness, non, geometry, :unknown, :rhs)
    function Problem(mat; thickness=1.0, type=:Solid, bandwidth=:none,
        nameTopSurface="", nameVolume="", dim::Int=3,
        field::Symbol=:unknown, rhs_field::Symbol=:rhs)
        if type == :dummy
            return new("dummy", :dummy, 0, 0, mat, 0, 0, LowLevelFEM.Geometry(), field, rhs_field)
        end
        pdim = 3
        dim0 = dim

        #if Sys.CPU_THREADS != Threads.nthreads()
        #    @warn "Number of threads($(Threads.nthreads())) ≠ logical threads in CPU($(Sys.CPU_THREADS))."
        #end
        geometry = LowLevelFEM.Geometry()

        if type == :Solid
            dim = 3
            pdim = 3
        elseif type == :PlaneStress
            dim = 2
            pdim = 2
        elseif type == :PlaneStrain
            dim = 2
            pdim = 2
        elseif type == :AxiSymmetric
            dim = 2
            pdim = 2
        elseif type == :PlaneHeatConduction
            dim = 2
            pdim = 1
        elseif type == :HeatConduction
            dim = 3
            pdim = 1
        elseif type == :AxiSymmetricHeatConduction
            dim = 2
            pdim = 1
        elseif type == :Truss
            dim = 3
            pdim = 3
        elseif type == :ScalarField
            dim0 < 1 && error("Problem: dimension of a :ScalarField problem must be one, two or three.")
            dim0 > 3 && error("Problem: dimension of a :ScalarField problem must be equal or less than three.")
            dim = dim0
            pdim = 1
        elseif type == :VectorField
            dim0 < 1 && error("Problem: dimension of a :VectorField problem must be one, two or three.")
            dim0 == 1 && error("Problem: dimension of a :VectorField problem must be greater than one.")
            dim0 > 3 && error("Problem: dimension of a :VectorField problem must be two or three.")
            dim = dim0
            pdim = dim0
        elseif type == :TensorField
            dim0 < 1 && error("Problem: dimension of a :TensorField problem must be one, two or three.")
            dim0 == 1 && error("Problem: dimension of a :TensorField problem must be greater than one.")
            dim0 > 3 && error("Problem: dimension of a :TensorField problem must be two or three.")
            dim = dim0
            pdim = 9
        elseif type == :Reynolds
            geometry = Geometry(nameTopSurface, nameVolume)
            dim = geometry.dim
            pdim = 1
        else
            error("Problem type can be: `:Solid`, `:PlaneStress`, `:PlaneStrain`, `:AxiSymmetric`, `:PlaneHeatConduction`, `:HeatConduction`, `:AxiSymmetricHeatConduction`, `:Truss`, `:ScalarField`, `VectorField`.
            Now problem type = $type ????")
        end
        if !isa(mat, Vector)
            error("Problem: materials are not arranged in a vector. Put them in [...]")
        end
        name = gmsh.model.getCurrent()
        gmsh.option.setString("General.GraphicsFontEngine", "Cairo")
        gmsh.option.setString("View.Format", "%.6g")

        material = mat

        if bandwidth ≠ :none
            elemTags = []
            for ipg in 1:length(material)
                phName = material[ipg].phName
                dimTags = gmsh.model.getEntitiesForPhysicalName(phName)
                for idm in 1:length(dimTags)
                    dimTag = dimTags[idm]
                    edim = dimTag[1]
                    etag = dimTag[2]
                    if edim != dim && (type != :Truss || edim != 1)
                        error("Problem: dimension of the problem ($dim) is different than the dimension of finite elements ($edim).")
                    end
                    elementTypes, elementTags, nodeTags = gmsh.model.mesh.getElements(edim, etag)
                    for i in 1:length(elementTags)
                        if length(elementTags[i]) == 0
                            error("Problem: No mesh in model '$name'.")
                        end
                        for j in 1:length(elementTags[i])
                            push!(elemTags, elementTags[i][j])
                        end
                    end
                end
            end

            if bandwidth != :RCMK && bandwidth != :Hilbert && bandwidth != :Metis && bandwidth != :none
                error("Problem: bandwidth can be `:Hilbert`, `:Metis`, `:RCMK` or `:none`. Now it is `$(bandwidth)`")
            end

            #method = bandwidth == :none ? :RCMK : bandwidth
            oldTags, newTags = gmsh.model.mesh.computeRenumbering(bandwidth, elemTags)
            #permOldTags = sortperm(oldTags)
            #sortNewTags = 1:length(oldTags)
            #newTags[permOldTags] = sortNewTags
            gmsh.model.mesh.renumberNodes(oldTags, newTags)
        end

        nodeTags, coord, parametricCoord = gmsh.model.mesh.getNodes()
        non = length(nodeTags)
        return new(name, type, dim, pdim, material, thickness, non, geometry, field, rhs_field)
    end
end


In [ ]:
struct BoundaryCondition
    phName::String
    problem::Union{Problem,Nothing}
    values::Dict{Symbol,Union{Number,Function,ScalarField}}

    function BoundaryCondition(phName::String; problem=nothing, kwargs...)
        vals = Dict{Symbol,Any}()
        for (k, v) in kwargs
            vals[k] = v
        end
        return new(phName, problem, vals)
    end
end

In [116]:
function BoundaryConditionFields(bc::BoundaryCondition)
    return keys(bc.values)
end

BoundaryConditionFields (generic function with 3 methods)

In [117]:
function parse_component(sym::Symbol, problem::Problem)
    s = String(sym)
    pf = String(problem.field)

    if !startswith(s, pf)
        return nothing
    end

    comp = s[length(pf)+1:end]   # lehet "", "x", "xy", stb.

    return comp
end

parse_component (generic function with 1 method)

In [ ]:
function component_index(problem::Problem, comp::String)

    if problem.type == :ScalarField
        comp == "" || error("Scalar field has no components.")
        return 1

    elseif problem.type == :VectorField
        comp in ("x", "y", "z") || error("Invalid vector component $comp")
        return findfirst(==(comp), ["x", "y", "z"])

    elseif problem.type == :TensorField
        comps = ["xx", "yx", "zx", "xy", "yy", "zy", "xz", "yz", "zz"]
        comp in comps || error("Invalid tensor component $comp")
        return findfirst(==(comp), comps)

    else
        error("Unsupported problem type for BC parsing.")
    end
end

component_index (generic function with 1 method)

In [ ]:
@inline function nodeToDof(problem::Problem, node::Int, comp::Int)
    return (node - 1) * problem.pdim + comp
end

nodeToDof (generic function with 1 method)

In [135]:
function nodesOnPhysicalGroup(problem::Problem, phName::String)

    dimTags = gmsh.model.getEntitiesForPhysicalName(phName)

    isempty(dimTags) && error("Physical group '$phName' not found.")

    nodes = Int[]

    for dimTag in dimTags
        edim = dimTag[1]
        etag = dimTag[2]

        # csak valódi perem entitás
        if edim < problem.dim
            elementTypes, elementTags, nodeTags =
                gmsh.model.mesh.getElements(edim, etag)

            for nt in nodeTags
                append!(nodes, nt)
            end
        else
            error("Cannot apply Dirichlet BC on entity of dimension $edim in $(problem.dim)D problem.")
        end
    end

    return unique(nodes)
end

nodesOnPhysicalGroup (generic function with 1 method)

In [ ]:
function constrainedDoFs(problem::Problem,
    supports::Vector{BoundaryCondition})

    dofs = Int[]

    for bc in supports

        # multi-field check
        if bc.problem !== nothing && bc.problem !== problem
            continue
        end

        node_ids = nodesOnPhysicalGroup(problem, bc.phName)

        for (sym, val) in bc.values

            comp = parse_component(sym, problem)
            comp === nothing && continue

            cidx = component_index(problem, comp)

            for node in node_ids
                push!(dofs, (node - 1) * problem.pdim + cidx)
            end
        end
    end

    return unique(dofs)
end

constrainedDoFs (generic function with 1 method)

In [ ]:
"""
    LoadCondition(phName; problem=nothing, kwargs...)

Generic load specification for assembling right-hand side vectors.

- `phName` is a Gmsh physical group name.
- `problem` optionally binds the load to a specific field (required for multi-field).
- `values` stores component-wise load data, such as `fx`, `fy`, `q`, `σxx`, etc.,
  as `Number`, `Function`, or `ScalarField`.

The interpretation (surface traction vs body force vs flux, etc.) is decided
by the assembly routine, not by this struct.
"""
struct LoadCondition
    phName::String
    problem::Union{Problem,Nothing}
    values::Dict{Symbol,Union{Number,Function,ScalarField}}

    function LoadCondition(phName::String; problem=nothing, kwargs...)
        vals = Dict{Symbol,Union{Number,Function,ScalarField}}()
        for (k, v) in kwargs
            vals[k] = v
        end
        return new(phName, problem, vals)
    end
end

LoadCondition

In [138]:
function loads_for_problem(loads::Vector{LoadCondition}, P::Problem)
    return filter(lc -> lc.problem === nothing || lc.problem === P, loads)
end

loads_for_problem (generic function with 1 method)

In [139]:
function loadVector(problem, loads; F=nothing)
    if problem.type == :dummy
        return nothing
    end
    
    gmsh.model.setCurrent(problem.name)
    if !isa(loads, Vector)
        error("loadVector: loads are not arranged in a vector. Put them in [...]")
    end

    Fe = nothing
    Fmap = nothing
    if F !== nothing
        Fe = nodesToElements(F)
        Fmap = Dict(zip(Fe.numElem, Fe.A))
    end

    pdim = problem.pdim
    DIM = problem.dim
    b = problem.thickness
    non = problem.non
    dof = pdim * non
    ncoord2 = zeros(3 * problem.non)
    f = nothing
    fp = zeros(dof,1)
    nsteps = 1
    for n in 1:length(loads)
        name = loads[n].phName
        fx = loads[n].fx
        fy = loads[n].fy
        fz = loads[n].fz
        T = loads[n].T
        p = loads[n].p
        hc = loads[n].h
        T0 = loads[n].T
        qn = loads[n].qn
        hs = loads[n].h

        q = loads[n].q

        fxy = loads[n].fxy
        fyz = loads[n].fyz
        fzx = loads[n].fzx
        
        fyx = loads[n].fyx
        fzy = loads[n].fzy
        fxz = loads[n].fxz

        fx  = (fx  isa ScalarField && isElementwise(fx))  ? elementsToNodes(fx)  : fx
        fy  = (fy  isa ScalarField && isElementwise(fy))  ? elementsToNodes(fy)  : fy
        fz  = (fz  isa ScalarField && isElementwise(fz))  ? elementsToNodes(fz)  : fz
        T   = (T   isa ScalarField && isElementwise(T))   ? elementsToNodes(T)   : T
        p   = (p   isa ScalarField && isElementwise(p))   ? elementsToNodes(p)   : p
        hc  = (hc  isa ScalarField && isElementwise(hc))  ? elementsToNodes(hc)  : hc
        T0  = (T0  isa ScalarField && isElementwise(T0))  ? elementsToNodes(T0)  : T0
        qn  = (qn  isa ScalarField && isElementwise(qn))  ? elementsToNodes(qn)  : qn
        hs  = (hs  isa ScalarField && isElementwise(hs))  ? elementsToNodes(hs)  : hs
        q   = (q   isa ScalarField && isElementwise(q))   ? elementsToNodes(q)   : q
        fxy = (fxy isa ScalarField && isElementwise(fxy)) ? elementsToNodes(fxy) : fxy
        fyz = (fyz isa ScalarField && isElementwise(fyz)) ? elementsToNodes(fyz) : fyz
        fzx = (fzx isa ScalarField && isElementwise(fzx)) ? elementsToNodes(fzx) : fzx
        fyx = (fyx isa ScalarField && isElementwise(fyx)) ? elementsToNodes(fyx) : fyx
        fzy = (fzy isa ScalarField && isElementwise(fzy)) ? elementsToNodes(fzy) : fzy
        fxz = (fxz isa ScalarField && isElementwise(fxz)) ? elementsToNodes(fxz) : fxz
    
        nsteps = 1
        for i in [fx, fy , fz, T,  p, hc, T0, qn, hs, q, fxy, fyz, fzx, fyx, fzy, fxz]
            if i isa ScalarField
                nsteps = max(nsteps, i.nsteps)
            end
        end
        if nsteps > size(fp, 2)
            fp = hcat(fp, zeros(dof, nsteps - size(fp,2)))
        end

        (qn !== nothing || hc !== nothing || hs !== nothing || T !== nothing) && (fx !== nothing || fy !== nothing || fz !== nothing) &&
            error("loadVector: qn/h/T∞ and fx/fy/fz cannot be defined in the same BC.")
        if pdim == 1 || pdim == 2 || pdim == 3 || pdim == 9
            f = zeros(pdim, nsteps)
        else
            error("loadVector: dimension of the problem is $(problem.dim).")
        end
        if p !== nothing && DIM == 3 && pdim == 3
            nv = -normalVector(problem, name)
            ex = VectorField(problem, [field(name, fx=1, fy=0, fz=0)])
            ey = VectorField(problem, [field(name, fx=0, fy=1, fz=0)])
            ez = VectorField(problem, [field(name, fx=0, fy=0, fz=1)])
            if p isa Number || p isa ScalarField
                fy = elementsToNodes((nv ⋅ ey) * p)
                fz = elementsToNodes((nv ⋅ ez) * p)
                fx = elementsToNodes((nv ⋅ ex) * p)
            elseif p isa Function
                pp = scalarField(problem, [field(name, f=p)])
                fy = elementsToNodes((nv ⋅ ey) * pp)
                fz = elementsToNodes((nv ⋅ ez) * pp)
                fx = elementsToNodes((nv ⋅ ex) * pp)
            end
        end
        if p !== nothing && DIM ≠ 3
            error("loadVector: pressure can be given on a surface of a 3D solid.")
        end
        fx = fx !== nothing ? fx : 0.0
        fy = fy !== nothing ? fy : 0.0
        fz = fz !== nothing ? fz : 0.0
        dimTags = gmsh.model.getEntitiesForPhysicalName(name)
        for i ∈ 1:length(dimTags)
            dimTag = dimTags[i]
            dim = dimTag[1]
            tag = dimTag[2]
            elementTypes, elementTags, elemNodeTags = gmsh.model.mesh.getElements(dim, tag)
            nodeTags::Vector{Int64}, ncoord, parametricCoord = gmsh.model.mesh.getNodes(dim, tag, true, false)
            ncoord2[nodeTags*3 .- 2] = ncoord[1:3:length(ncoord)]
            ncoord2[nodeTags*3 .- 1] = ncoord[2:3:length(ncoord)]
            ncoord2[nodeTags*3 .- 0] = ncoord[3:3:length(ncoord)]
            for ii in 1:length(elementTypes)
                elementName, dim, order, numNodes::Int64, localNodeCoord, numPrimaryNodes = gmsh.model.mesh.getElementProperties(elementTypes[ii])
                nnoe = reshape(elemNodeTags[ii], numNodes, :)'
                intPoints, intWeights = gmsh.model.mesh.getIntegrationPoints(elementTypes[ii], "Gauss" * string(2order+1))
                numIntPoints = length(intWeights)
                comp, fun, ori = gmsh.model.mesh.getBasisFunctions(elementTypes[ii], intPoints, "Lagrange")
                h = reshape(fun, :, numIntPoints)
                nnet = zeros(Int, length(elementTags[ii]), numNodes)
                H = zeros(pdim * numIntPoints, pdim * numNodes)
                for j in 1:numIntPoints
                    for k in 1:numNodes
                        for l in 1:pdim
                            H[j*pdim-(pdim-l), k*pdim-(pdim-l)] = h[k, j]
                        end
                    end
                end
                f1 = zeros(pdim * numNodes, nsteps)
                nn2 = zeros(Int, pdim * numNodes)
                @inbounds for l in 1:length(elementTags[ii])
                    elem = elementTags[ii][l]
                    for k in 1:numNodes
                        nnet[l, k] = elemNodeTags[ii][(l-1)*numNodes+k]
                    end
                    jac, jacDet, coord = gmsh.model.mesh.getJacobian(elem, intPoints)
                    Jac = reshape(jac, 3, :)
                    fill!(f1, 0.0)
                    @inbounds for j in 1:numIntPoints
                        x = h[:, j]' * ncoord2[nnet[l, :] * 3 .- 2]
                        y = h[:, j]' * ncoord2[nnet[l, :] * 3 .- 1]
                        z = h[:, j]' * ncoord2[nnet[l, :] * 3 .- 0]
                        if hc !== nothing && T0 !== nothing
                            if hc isa Function
                                error("heatConvectionVector: h cannot be a function.")
                            end
                            f[1,:] .= _loadvec_helper(T0, h, x, y, z, nnet, j, l, nsteps) * hc
                        elseif qn !== nothing
                            f[1,:] .= _loadvec_helper(qn, h, x, y, z, nnet, j, l, nsteps)
                        elseif hs !== nothing
                            f[1,:] .= _loadvec_helper(hs, h, x, y, z, nnet, j, l, nsteps)
                        elseif q !== nothing
                            f[1,:] .= _loadvec_helper(q, h, x, y, z, nnet, j, l, nsteps)
                        elseif fx !== nothing && pdim <= 3
                            f[1,:] .= _loadvec_helper(fx, h, x, y, z, nnet, j, l, nsteps)
                        end
                        if pdim > 1 && pdim <= 3
                            f[2,:] .= _loadvec_helper(fy, h, x, y, z, nnet, j, l, nsteps)
                        end
                        if pdim == 3
                            f[3,:] .= _loadvec_helper(fz, h, x, y, z, nnet, j, l, nsteps)
                        end
                        if pdim == 9
                            fill!(f, 0.0)
                            if fx !== nothing
                                f[1,:] .= _loadvec_helper(fx, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fy !== nothing
                                f[5,:] .= _loadvec_helper(fy, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fz !== nothing
                                f[9,:] .= _loadvec_helper(fz, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fxy !== nothing
                                f[4,:] .= _loadvec_helper(fxy, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fyz !== nothing
                                f[8,:] .= _loadvec_helper(fyz, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fzx !== nothing
                                f[3,:] .= _loadvec_helper(fzx, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fyx !== nothing
                                f[2,:] .= _loadvec_helper(fyx, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fzy !== nothing
                                f[6,:] .= _loadvec_helper(fzy, h, x, y, z, nnet, j, l, nsteps)
                            end
                            if fxz !== nothing
                                f[7,:] .= _loadvec_helper(fxz, h, x, y, z, nnet, j, l, nsteps)
                            end
                        end

                        # -------- FOLLOWER LOAD (Total Lagrange, Piola) ----------
                        if F !== nothing
                            # F elemértékek
                            Fnode = Fmap[elem]   # 9*numNodes × 1

                            # F interpoláció Gauss-pontra (pont úgy, ahogy Kext-ben)
                            Fgp = zeros(DIM, DIM)
                            for a in 1:numNodes
                                Na = h[a, j]
                                Fgp .+= Na * reshape(Fnode[9a-8:9a], DIM, DIM)
                            end

                            Rgp = _rotation_from_F(Fgp)

                            Jgp = det(Fgp)
                            FinvT = inv(Fgp)'

                            if !(Jgp > 0)
                                error("Element inversion detected: det(Fgp) = $Jgp at elem=$elem, gp=$j")
                            end

                            # reference surface normal at GP from reference Jacobian
                            t1 = Jac[:, 3*j-2]          # first tangent in reference
                            t2 = Jac[:, 3*j-1]          # second tangent in reference
                            nref = cross(t1, t2)
                            Ja_ref = norm(nref)         # ugyanaz, mint amit lent Ja-ként használsz
                            N0 = nref / Ja_ref          # unit normal in reference

                            # Nanson scalar: da = |J F^{-T} N0| dA
                            scale = norm(Jgp * FinvT * N0)

                            # Piola transzformáció a traction-re
                            for s in 1:nsteps
                                ##f[:,s] .= Jgp * FinvT * f[:,s]
                                #f[:, s] .= Rgp * f[:, s]          # t = R * t_loc  (itt f a traction, lokálisnak tekint
                                #f[:, s] .= Jgp * FinvT * f[:, s]  # Piola: t0 = J F^{-T} t
                                #f[:, s] .= Rgp * f[:, s]        # true follower: rotate traction with body
                                #f[:, s] .*= scale               # ONLY area scaling (no FinvT acting on t)
                                f[:, s] .= Fgp * f[:, s]        # true follower: rotate traction with body
                            end
                        end
                        # ---------------------------------------------------------

                        r = x
                        H1 = H[j*pdim-(pdim-1):j*pdim, 1:pdim*numNodes] # H1[...] .= H[...] ????
                        ############### NANSON ######## 3D ###################################
                        if DIM == 3 && dim == 3
                            Ja = jacDet[j]
                        elseif DIM == 3 && dim == 2
                            xy = Jac[1, 3*j-2] * Jac[2, 3*j-1] - Jac[2, 3*j-2] * Jac[1, 3*j-1]
                            yz = Jac[2, 3*j-2] * Jac[3, 3*j-1] - Jac[3, 3*j-2] * Jac[2, 3*j-1]
                            zx = Jac[3, 3*j-2] * Jac[1, 3*j-1] - Jac[1, 3*j-2] * Jac[3, 3*j-1]
                            Ja = √(xy^2 + yz^2 + zx^2)
                        elseif DIM == 3 && dim == 1
                            Ja = √((Jac[1, 3*j-2])^2 + (Jac[2, 3*j-2])^2 + (Jac[3, 3*j-2])^2)
                        elseif DIM == 3 && dim == 0
                            Ja = 1
                            ############ 2D #######################################################
                        elseif DIM == 2 && dim == 2 && problem.type != :AxiSymmetric && problem.type != :AxiSymmetricHeatConduction
                            Ja = jacDet[j] * b
                        elseif DIM == 2 && dim == 2 && (problem.type == :AxiSymmetric || problem.type == :AxiSymmetricHeatConduction)
                            Ja = 2π * jacDet[j] * r
                        elseif DIM == 2 && dim == 1 && problem.type != :AxiSymmetric && problem.type != :AxiSymmetricHeatConduction
                            Ja = √((Jac[1, 3*j-2])^2 + (Jac[2, 3*j-2])^2) * b
                        elseif DIM == 2 && dim == 1 && (problem.type == :AxiSymmetric || problem.type == :AxiSymmetricHeatConduction)
                            Ja = 2π * √((Jac[1, 3*j-2])^2 + (Jac[2, 3*j-2])^2) * r
                        elseif DIM == 2 && dim == 0
                            Ja = 1
                            ############ 1D #######################################################
                        elseif DIM == 1 && dim == 1
                            Ja = Jac[1, 3*j-2] * b
                        elseif DIM == 1 && dim == 0
                            Ja = 1
                        else
                            error("loadVector: dimension of the problem is $(problem.dim), dimension of load is $dim.")
                        end
                        f1 += H1' * f * Ja * intWeights[j]
                    end
                    for k in 1:pdim
                        nn2[k:pdim:pdim*numNodes] = pdim * nnoe[l, 1:numNodes] .- (pdim - k)
                    end
                    fp[nn2,1:nsteps] .+= f1
                end
            end
        end
    end
    type = :null
    if pdim == 3
        type = :v3D
    elseif pdim == 2
        type = :v2D
    elseif pdim == 1
        type = :scalar
    elseif pdim == 9
        type = :tensor
    else
        error("loadVector: wrong pdim ($pdim).")
    end
    ts = [i for i in 0:nsteps-1]
    if type == :v3D || type == :v2D
        return VectorField([], reshape(fp, :, nsteps), ts, [], nsteps, type, problem)

    elseif type == :scalar
        return ScalarField([], reshape(fp, :, nsteps), ts, [], nsteps, type, problem)

    elseif type == :tensor
        return TensorField([], reshape(fp, :, nsteps), ts, [], nsteps, type, problem)
    end
end

loadVector (generic function with 1 method)

In [93]:
"""
    SystemMatrix

Unified system matrix type for single-field, operator, and multi-field
(block) problems.

# Fields
- `A::SparseMatrixCSC{Float64,Int}`
    Sparse matrix of the operator/system.

Single-operator metadata (nxm case):
- `model::Union{Problem,Nothing}`
    Trial space (column space).
- `test_model::Union{Problem,Nothing}`
    Test space (row space).

Multi-field (block system) metadata:
- `problems::Union{Vector{Problem},Nothing}`
    Ordered list of Problems in the global block system.
- `offsets::Union{Vector{Int},Nothing}`
    Starting global DOF indices of each Problem.

# Semantics
- If `problems === nothing` → single operator case.
- If `problems !== nothing` → multi-field block system.
"""
struct SystemMatrix
    A::SparseMatrixCSC{Float64,Int}

    # ---- Single operator metadata ----
    model::Union{Problem,Nothing}        # trial space
    test_model::Union{Problem,Nothing}   # test space

    # ---- Block system metadata ----
    problems::Union{Vector{Problem},Nothing}
    offsets::Union{Vector{Int},Nothing}

    SystemMatrix(A::SparseMatrixCSC{Float64,Int},
        model::Problem) =
        SystemMatrix(A, model, model, nothing, nothing)

    SystemMatrix(A::SparseMatrixCSC{Float64,Int},
        model::Problem,
        test_model::Problem) =
        SystemMatrix(A, model, test_model, nothing, nothing)

    SystemMatrix(A::SparseMatrixCSC{Float64,Int},
        problems::Vector{Problem},
        offsets::Vector{Int}) =
        SystemMatrix(A, nothing, nothing, problems, offsets)

    function SystemMatrix(blocks::Matrix{SystemMatrix})

        nrows, ncols = size(blocks)

        # ----------------------------------------------------------
        # 1) Collect unique Problems (first appearance rule)
        # ----------------------------------------------------------
        problems = Problem[]

        function add_problem!(P)
            P === nothing && return
            if all(q -> q !== P, problems)
                push!(problems, P)
            end
        end

        for i in 1:nrows
            for j in 1:ncols
                blk = blocks[i, j]
                add_problem!(blk.test_model)
                add_problem!(blk.model)
            end
        end

        isempty(problems) && error("No valid Problems found in block matrix.")

        # ----------------------------------------------------------
        # 2) Compute global offsets
        # ----------------------------------------------------------
        offsets = Vector{Int}(undef, length(problems))
        offsets[1] = 0
        for i in 2:length(problems)
            offsets[i] = offsets[i-1] + ndofs(problems[i-1])
        end

        # helper: find index of problem
        function problem_index(P)
            for (k, q) in enumerate(problems)
                if q === P
                    return k
                end
            end
            error("Problem not found in block metadata.")
        end

        # ----------------------------------------------------------
        # 3) Assemble global sparse matrix
        # ----------------------------------------------------------
        I = Int[]
        J = Int[]
        V = Float64[]

        for bi in 1:nrows
            for bj in 1:ncols

                blk = blocks[bi, bj]

                blk.A === nothing && continue
                isempty(blk.A) && continue

                rowP = blk.test_model
                colP = blk.model

                rowP === nothing && error("Block missing test_model.")
                colP === nothing && error("Block missing model.")

                iP = problem_index(rowP)
                jP = problem_index(colP)

                row_offset = offsets[iP]
                col_offset = offsets[jP]

                # extract local indices
                rows, cols, vals = findnz(blk.A)

                for k in eachindex(vals)
                    push!(I, row_offset + rows[k])
                    push!(J, col_offset + cols[k])
                    push!(V, vals[k])
                end
            end
        end

        total_dofs = offsets[end] + ndofs(problems[end])

        A_big = sparse(I, J, V, total_dofs, total_dofs)

        # ----------------------------------------------------------
        # 4) Return multi-field SystemMatrix
        # ----------------------------------------------------------
        return SystemMatrix(A_big, nothing, nothing, problems, offsets)
    end
end


SystemMatrix

In [94]:
struct BoundaryCondition
    phName::String

    # Dirichlet-type (primary variables)
    ux::Union{Nothing,Number,Function,ScalarField}
    uy::Union{Nothing,Number,Function,ScalarField}
    uz::Union{Nothing,Number,Function,ScalarField}
    p::Union{Nothing,Number,Function,ScalarField}   # pressure / scalar field
    T::Union{Nothing,Number,Function,ScalarField}   # temperature

    # Neumann-type (fluxes / forces)
    f::Union{Nothing,Number,Function,ScalarField}
    fx::Union{Nothing,Number,Function,ScalarField}
    fy::Union{Nothing,Number,Function,ScalarField}
    fz::Union{Nothing,Number,Function,ScalarField}
    fxy::Union{Nothing,Number,Function,ScalarField}
    fyz::Union{Nothing,Number,Function,ScalarField}
    fzx::Union{Nothing,Number,Function,ScalarField}
    fyx::Union{Nothing,Number,Function,ScalarField}
    fzy::Union{Nothing,Number,Function,ScalarField}
    fxz::Union{Nothing,Number,Function,ScalarField}
    #qx ::Union{Nothing, Number, Function}   # heat flux / generic scalar flux
    #qy ::Union{Nothing, Number, Function}   # heat flux / generic scalar flux
    #qz ::Union{Nothing, Number, Function}   # heat flux / generic scalar flux
    kx::Union{Nothing,Number,Function}
    ky::Union{Nothing,Number,Function}
    kz::Union{Nothing,Number,Function}
    q::Union{Nothing,Number,Function,ScalarField}   # heat flux / generic scalar flux
    qn::Union{Nothing,Number,Function,ScalarField}   # heat flux / generic scalar flux
    h::Union{Nothing,Number,Function,ScalarField}

    # Stress / traction components
    sx::Union{Nothing,Number,Function,ScalarField}
    sy::Union{Nothing,Number,Function,ScalarField}
    sz::Union{Nothing,Number,Function,ScalarField}
    sxy::Union{Nothing,Number,Function,ScalarField}
    syz::Union{Nothing,Number,Function,ScalarField}
    szx::Union{Nothing,Number,Function,ScalarField}

    CS::Union{Nothing,CoordinateSystem}
    #Q::Union{Nothing,Transformation}

    field::Union{Nothing,Problem}

    function BoundaryCondition(phName::String; kwargs...)
        fields = fieldnames(BoundaryCondition)
        values = map(fields) do f
            f == :phName ? phName : get(kwargs, f, nothing)
        end
        return new(values...)
    end
end


In [95]:
abstract type AbstractOp end

struct IdOp <: AbstractOp end     # u
struct GradOp <: AbstractOp end     # ∇u
struct DivOp <: AbstractOp end     # div u
struct CurlOp <: AbstractOp end    # rot u
struct SymGradOp <: AbstractOp end
struct TensorDivOp <: AbstractOp end
struct AdvOp <: AbstractOp
    b::NTuple{3,Float64}
end

In [96]:
"""
    ndofs(problem::Problem)

Return total number of dofs for a single-field problem.
"""
ndofs(P::Problem) = P.non * P.pdim

"""
    op_outdim(op, P)

Return the number of components produced by applying `op` to a field of `P`.
This determines the row-count of the operator matrix at a Gauss point.
"""
function op_outdim(::IdOp, P::Problem)
    return P.pdim
end

function op_outdim(::GradOp, P::Problem)
    # Scalar: grad -> dim
    if P.pdim == 1
        return P.dim
    end
    # Vector: grad(u) -> dim×pdim (full gradient, column per component)
    return P.dim * P.pdim
end

function op_outdim(::DivOp, P::Problem)
    # Vector: div -> 1 (assume pdim==dim)
    @assert P.pdim == P.dim "DivOp currently assumes vector field with pdim == dim."
    return 1
end

function op_outdim(::CurlOp, P::Problem)
    @assert P.pdim == P.dim "CurlOp requires vector field with pdim == dim."
    return (P.dim == 2) ? 1 : 3
end

function op_outdim(::SymGradOp, P::Problem)
    @assert P.pdim == P.dim "SymGradOp requires vector field with pdim == dim."
    return (P.dim == 2) ? 3 : 6  # engineering strain components
end

function op_outdim(::TensorDivOp, P::Problem)
    @assert P.pdim == P.dim^2 "TensorDivOp requires pdim == dim^2 (full 2nd-order tensor)."
    return P.dim
end

function op_outdim(op::AdvOp, P::Problem)
    @assert P.pdim == 1  # scalar field
    return 1
end

op_outdim (generic function with 7 methods)

In [97]:
"""
    build_B!(B, op, P, k, h, ∂h, numNodes)

Fill the operator matrix `B` at Gauss point `k` for operator `op` and problem `P`.
- `B` has size (op_outdim(op,P), P.pdim*numNodes)
"""
function build_B!(B::AbstractMatrix, ::IdOp, P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    pdim = P.pdim
    @inbounds for a in 1:numNodes
        Na = h[(k-1)*numNodes+a]
        @inbounds for c in 1:pdim
            row = c
            col = (a - 1) * pdim + c
            B[row, col] = Na
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, ::GradOp, P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    pdim = P.pdim
    dim = P.dim

    if pdim == 1
        # grad(scalar): rows = dim
        @inbounds for a in 1:numNodes
            col = a # scalar: (a-1)*1 + 1
            @inbounds for d in 1:dim
                row = d
                B[row, col] = ∂h[d, (k-1)*numNodes+a]
            end
        end
    else
        # grad(vector): rows = dim*pdim
        # ordering rows: (comp-1)*dim + d   (component-major)
        @inbounds for a in 1:numNodes
            @inbounds for c in 1:pdim
                col = (a - 1) * pdim + c
                @inbounds for d in 1:dim
                    row = (c - 1) * dim + d
                    B[row, col] = ∂h[d, (k-1)*numNodes+a]
                end
            end
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, ::DivOp, P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    pdim = P.pdim
    dim = P.dim
    @assert pdim == dim

    # div(u) = Σ_i ∂u_i/∂x_i
    # For basis (node a, component i): contribution is ∂N_a/∂x_i
    @inbounds for a in 1:numNodes
        @inbounds for i in 1:dim
            col = (a - 1) * pdim + i
            B[1, col] += ∂h[i, (k-1)*numNodes+a]
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, ::CurlOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    dim = P.dim
    pdim = P.pdim
    @assert pdim == dim

    if dim == 2
        # curl(u) = ∂uy/∂x - ∂ux/∂y  (scalar)
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            B[1, colx] = -∂h[2, (k-1)*numNodes+a]   # -∂N/∂y * ux_a
            B[1, coly] = ∂h[1, (k-1)*numNodes+a]   #  ∂N/∂x * uy_a
        end
    else
        # 3D curl:
        # cx = ∂uz/∂y - ∂uy/∂z
        # cy = ∂ux/∂z - ∂uz/∂x
        # cz = ∂uy/∂x - ∂ux/∂y
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            colz = (a - 1) * pdim + 3

            dNx = ∂h[1, (k-1)*numNodes+a]
            dNy = ∂h[2, (k-1)*numNodes+a]
            dNz = ∂h[3, (k-1)*numNodes+a]

            # cx row = 1
            B[1, coly] = -dNz
            B[1, colz] = dNy

            # cy row = 2
            B[2, colx] = dNz
            B[2, colz] = -dNx

            # cz row = 3
            B[3, colx] = -dNy
            B[3, coly] = dNx
        end
    end

    return B
end

function build_B!(B::AbstractMatrix, ::SymGradOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    dim = P.dim
    pdim = P.pdim
    @assert pdim == dim

    if dim == 2
        # rows: [εxx, εyy, γxy]
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            dNx = ∂h[1, (k-1)*numNodes+a]
            dNy = ∂h[2, (k-1)*numNodes+a]

            B[1, colx] = dNx          # εxx
            B[2, coly] = dNy          # εyy
            B[3, colx] = dNy          # γxy = dux/dy + duy/dx
            B[3, coly] = dNx
        end
    else
        # rows: [εxx, εyy, εzz, γxy, γxz, γyz]
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            colz = (a - 1) * pdim + 3

            dNx = ∂h[1, (k-1)*numNodes+a]
            dNy = ∂h[2, (k-1)*numNodes+a]
            dNz = ∂h[3, (k-1)*numNodes+a]

            B[1, colx] = dNx          # εxx
            B[2, coly] = dNy          # εyy
            B[3, colz] = dNz          # εzz

            B[4, colx] = dNy          # γxy
            B[4, coly] = dNx

            B[5, colx] = dNz          # γxz
            B[5, colz] = dNx

            B[6, coly] = dNz          # γyz
            B[6, colz] = dNy
        end
    end

    return B
end

function build_B!(B::AbstractMatrix, ::TensorDivOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    dim = P.dim
    pdim = P.pdim  # dim^2

    # assume tensor component ordering at node:
    # σ_ij mapped to α = (i-1)*dim + j   (i=row, j=col)
    @inbounds for a in 1:numNodes
        for i in 1:dim
            row = i
            for j in 1:dim
                col = (a - 1) * pdim + (i - 1) * dim + j
                B[row, col] = ∂h[j, (k-1)*numNodes+a]  # ∂/∂x_j
            end
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, op::AdvOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)

    fill!(B, 0.0)
    b = op.b
    dim = P.dim

    @inbounds for a in 1:numNodes
        val = 0.0
        for d in 1:dim
            val += b[d] * ∂h[d, (k-1)*numNodes+a]
        end
        B[1, a] = val
    end

    return B
end

build_B! (generic function with 7 methods)

In [98]:
@inline function _build_elemwise_coeff_dict(coef::ScalarField)
    p = nodesToElements(coef)
    return Dict(zip(p.numElem, p.A))  # elemTag => coeff nodal vector(s)
end

@inline function _coeff_at_gp(pa::Dict{<:Integer,<:AbstractMatrix}, elem::Integer, hcol::AbstractVector)
    return dot(view(pa[elem], :, 1), hcol)
end


_coeff_at_gp (generic function with 1 method)

In [99]:
"""
    assemble_operator(Pu, op_u, Ps, op_s; coefficient=1.0)

Assemble a sparse block matrix for the bilinear form
    ∫ (Op_s v) · (Op_u u) * coefficient dΩ
where trial field is from problem `Pu` and test field is from problem `Ps`.

Currently supports scalar coefficient (Number or ScalarField).
Requires that both problems share the same gmsh model (same mesh).
"""
function assemble_operator(
    Pu::Problem, op_u::AbstractOp,
    Ps::Problem, op_s::AbstractOp;
    coefficient::Union{Number,ScalarField}=1.0,
    weight=nothing)

    @assert Pu.name == Ps.name "Both problems must refer to the same gmsh model/mesh."
    @assert Pu.dim == Ps.dim "Both problems must have the same spatial dimension."
    gmsh.model.setCurrent(Pu.name)

    # output dims must match for the dot-product integrand
    out_u = op_outdim(op_u, Pu)
    out_s = op_outdim(op_s, Ps)
    @assert out_u == out_s "Operator output dims mismatch: $out_u vs $out_s."

    # coefficient prep
    pconst = 1.0
    pa = Dict{Int,Any}()
    if coefficient isa Number
        pconst = Float64(coefficient)
    else
        pa = _build_elemwise_coeff_dict(coefficient)
    end

    # estimate IJV length: crude but OK for notebook prototype
    # worst-case: full dense per element: (pdim_s*numNodes)*(pdim_u*numNodes)
    lengthOfIJV = LowLevelFEM.estimateLengthOfIJV(Pu) * max(1, Ps.pdim) * max(1, Pu.pdim)
    I = Vector{Int}(undef, lengthOfIJV)
    J = Vector{Int}(undef, lengthOfIJV)
    V = Vector{Float64}(undef, lengthOfIJV)
    pos = 1

    dim = Pu.dim

    # loop materials in Pu (same physical names expected)
    for mat in Pu.material
        phName = mat.phName
        dimTags = gmsh.model.getEntitiesForPhysicalName(phName)

        for (edim, etag) in dimTags
            elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(edim, etag)

            for itype in eachindex(elemTypes)
                et = elemTypes[itype]
                _, _, order, numNodes::Int64, _, _ = gmsh.model.mesh.getElementProperties(et)

                intPoints, intWeights = gmsh.model.mesh.getIntegrationPoints(et, "Gauss" * string(2order + 1))
                numIntPoints = length(intWeights)

                _, fun, _ = gmsh.model.mesh.getBasisFunctions(et, intPoints, "Lagrange")
                h = reshape(fun, :, numIntPoints)

                _, dfun, _ = gmsh.model.mesh.getBasisFunctions(et, intPoints, "GradLagrange")
                ∇h = reshape(dfun, :, numIntPoints)

                # buffers
                nel = length(elemTags[itype])
                nnet = zeros(Int, nel, numNodes)
                invJac = zeros(3, 3numIntPoints)
                ∂h = zeros(dim, numNodes * numIntPoints)

                ndofs_u_loc = Pu.pdim * numNodes
                ndofs_s_loc = Ps.pdim * numNodes

                Bu = zeros(out_u, ndofs_u_loc)
                Bs = zeros(out_s, ndofs_s_loc)
                Ke = zeros(ndofs_s_loc, ndofs_u_loc)

                # connectivity table
                @inbounds for e in 1:nel
                    for a in 1:numNodes
                        nnet[e, a] = elemNodeTags[itype][(e-1)*numNodes+a]
                    end
                end

                tmpBu = similar(Bu)
                # element loop
                @inbounds for e in 1:nel
                    elem = elemTags[itype][e]

                    jac, jacDet, _ = gmsh.model.mesh.getJacobian(elem, intPoints)
                    Jac = reshape(jac, 3, :)

                    @inbounds for k in 1:numIntPoints
                        invJac[1:3, 3k-2:3k] .= inv(Jac[1:3, 3k-2:3k])'
                    end

                    # physical gradients of basis
                    fill!(∂h, 0.0)
                    @inbounds for k in 1:numIntPoints, a in 1:numNodes
                        invJk = invJac[1:dim, 3k-2:3k-(3-dim)]
                        gha = ∇h[a*3-2:a*3-(3-dim), k]
                        ∂h[1:dim, (k-1)*numNodes+a] .= invJk * gha
                    end

                    fill!(Ke, 0.0)

                    # integrate
                    @inbounds for k in 1:numIntPoints
                        cval = (coefficient isa ScalarField) ? _coeff_at_gp(pa, elem, view(h, :, k)) : pconst
                        w = jacDet[k] * intWeights[k] * cval

                        build_B!(Bu, op_u, Pu, k, h, ∂h, numNodes)
                        # --- after build_B!(Bu, ...)
                        # weight: nothing / Vector / Matrix
                        if weight !== nothing
                            if weight isa AbstractVector
                                @inbounds for r in 1:size(Bu, 1)
                                    wr = weight[r]
                                    @inbounds for c in 1:size(Bu, 2)
                                        Bu[r, c] *= wr
                                    end
                                end
                            else
                                # weight isa AbstractMatrix: Bu = weight * Bu
                                # buffer needed: tmp has same size as Bu
                                mul!(tmpBu, weight, Bu)
                                Bu .= tmpBu
                            end
                        end

                        #if weight !== nothing
                        #    @inbounds for r in 1:size(Bu, 1)
                        #        wr = weight[r]
                        #        @inbounds for c in 1:size(Bu, 2)
                        #            Bu[r, c] *= wr
                        #        end
                        #    end
                        #end
                        build_B!(Bs, op_s, Ps, k, h, ∂h, numNodes)

                        # Ke += Bs' * Bu * w
                        mul!(Ke, transpose(Bs), Bu, w, 1.0)
                    end

                    # scatter Ke(s,u) -> global IJV
                    @inbounds for a_loc in 1:ndofs_s_loc
                        node_a = div(a_loc - 1, Ps.pdim) + 1
                        comp_a = mod(a_loc - 1, Ps.pdim) + 1
                        Ia_node = nnet[e, node_a]
                        Ia = (Ia_node - 1) * Ps.pdim + comp_a

                        @inbounds for b_loc in 1:ndofs_u_loc
                            node_b = div(b_loc - 1, Pu.pdim) + 1
                            comp_b = mod(b_loc - 1, Pu.pdim) + 1
                            Jb_node = nnet[e, node_b]
                            Jb = (Jb_node - 1) * Pu.pdim + comp_b

                            if pos > length(I)
                                # grow (not pretty, but notebook-friendly)
                                newlen = Int(ceil(1.5length(I))) + 1000
                                resize!(I, newlen)
                                resize!(J, newlen)
                                resize!(V, newlen)
                            end

                            I[pos] = Ia
                            J[pos] = Jb
                            V[pos] = Ke[a_loc, b_loc]
                            pos += 1
                        end
                    end
                end
            end
        end
    end

    resize!(I, pos - 1)
    resize!(J, pos - 1)
    resize!(V, pos - 1)
    K = sparse(I, J, V, ndofs(Ps), ndofs(Pu))
    dropzeros!(K)
    return K
end


assemble_operator

In [100]:
"""
    compliance9_iso(E, nu; penalty=1e8)

Return a 9×9 compliance-like matrix acting on vec(σ) with ordering
(11,12,13,21,22,23,31,32,33). Symmetric part follows isotropic linear elasticity,
antisymmetric part is penalized with `penalty`.
"""
function compliance9_iso(E, nu; penalty=1e8)
    G = E / (2 * (1 + nu))

    # Voigt compliance (engineering shear):
    # [ε11, ε22, ε33, γ12, γ13, γ23] = S6 * [σ11, σ22, σ33, σ12, σ13, σ23]
    S6 = zeros(6, 6)
    S6[1, 1] = 1 / E
    S6[2, 2] = 1 / E
    S6[3, 3] = 1 / E
    S6[1, 2] = -nu / E
    S6[1, 3] = -nu / E
    S6[2, 1] = -nu / E
    S6[2, 3] = -nu / E
    S6[3, 1] = -nu / E
    S6[3, 2] = -nu / E
    S6[4, 4] = 1 / G
    S6[5, 5] = 1 / G
    S6[6, 6] = 1 / G

    # Map 9 -> 6 (take symmetric + engineering shear)
    # v9 = [σ11 σ12 σ13 σ21 σ22 σ23 σ31 σ32 σ33]
    # v6 = [σ11, σ22, σ33, σ12, σ13, σ23] with σ12 := (σ12+σ21)/2 etc.
    P = zeros(6, 9)
    # normals
    P[1, 1] = 1.0       # σ11
    P[2, 5] = 1.0       # σ22
    P[3, 9] = 1.0       # σ33
    # shear sym averages
    P[4, 2] = 0.5
    P[4, 4] = 0.5   # σ12
    P[5, 3] = 0.5
    P[5, 7] = 0.5   # σ13
    P[6, 6] = 0.5
    P[6, 8] = 0.5   # σ23

    # Sym part compliance in 9-space: Ssym = P' * S6 * P
    Ssym = P' * S6 * P

    # Antisym penalty: penalize (σ12-σ21), (σ13-σ31), (σ23-σ32)
    # Add penalty * Q'Q where Q*v9 = [σ12-σ21, σ13-σ31, σ23-σ32]
    Q = zeros(3, 9)
    Q[1, 2] = 1.0
    Q[1, 4] = -1.0
    Q[2, 3] = 1.0
    Q[2, 7] = -1.0
    Q[3, 6] = 1.0
    Q[3, 8] = -1.0
    Santi = penalty * (Q' * Q)

    return Ssym + Santi
end


compliance9_iso

In [101]:
gmsh.initialize()

In [102]:
structured_box_mesh()

In [103]:
material = Material("volume")

Pp = Problem([material], type=:ScalarField, field=:p)
Pu = Problem([material], type=:VectorField, field=:u)
Ps = Problem([material], type=:TensorField, field=:s)

Problem("", :TensorField, 3, 9, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :s, :rhs)

In [104]:
pres = BoundaryCondition("left", problem=Pp, p=2)

BoundaryCondition("left", nothing, nothing, nothing, 2, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing)

In [105]:
parse_component(:p, Pp)

""

In [133]:
supp = BoundaryCondition("left", problem=Pu, uz=2, uy=1)

BoundaryCondition("left", Problem("", :VectorField, 3, 3, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), Dict{Symbol, Any}(:uz => 2, :uy => 1))

In [107]:
parse_component(:uy, Pu)

"y"

In [127]:
supp = BoundaryCondition("left", problem=Ps, sxy=1)

BoundaryCondition("left", Problem("", :TensorField, 3, 9, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :s, :rhs), Dict{Symbol, Any}(:sxy => 1))

In [113]:
parse_component(:sxy, Ps)

"xy"

In [122]:
cmp = parse_component(:uy, Pu)
component_index(Pu, cmp)

2

In [124]:
cmp = parse_component(:syx, Ps)
component_index(Ps, cmp)

7

In [136]:
constrainedDoFs(Pu, [supp])

242-element Vector{Int64}:
   6
  27
 351
 108
 354
 111
 357
 114
 360
 117
 363
 120
 366
   ⋮
 104
   2
  53
  56
  59
  62
  65
  68
  71
  74
  77
   8